# Financial Sentiment Analysis: AAPL News Sentiment Using FinBERT

**Sentiment Dynamics in AAPL Financial News Using FinBERT**

This notebook implements an end-to-end sentiment analysis pipeline to quantify and interpret Apple Inc. (AAPL)'s market mood using financial news data. The analysis follows a Delta Lakehouse architecture (Bronze → Silver → Gold) and uses FinBERT for sentiment classification.

## Project Structure
1. **Data Ingestion** (Bronze Layer): Load raw JSON data
2. **Data Curation** (Silver Layer): Clean, deduplicate, and prepare data
3. **FinBERT Sentiment Inference**: Generate sentiment scores
4. **Daily Aggregation** (Gold Layer): Aggregate sentiment metrics
5. **Evaluation**: Correlation and co-movement analysis


In [ ]:
# Ensure required libraries are installed (safe to run multiple times)
%pip install -q transformers torch matplotlib seaborn scipy

print("Required libraries are installed.")


: 

In [ ]:
# Import required libraries
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For sentiment analysis
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")


## 1. Data Ingestion (Bronze Layer)

Load raw JSON data from Benzinga API containing AAPL news articles and historical price data.


In [ ]:
# Load news data
with open('../data/aapl_news.json', 'r', encoding='utf-8') as f:
    news_data = json.load(f)

# Load price data
with open('../data/aapl_price.json', 'r', encoding='utf-8') as f:
    price_data = json.load(f)

print(f"News articles loaded: {len(news_data)}")
print(f"Price records loaded: {len(price_data)}")
print(f"\nNews sample keys: {list(news_data[0].keys())}")
print(f"Price sample keys: {list(price_data[0].keys())}")


In [ ]:
# Convert to DataFrames (Bronze Layer)
df_news_bronze = pd.DataFrame(news_data)
df_price_bronze = pd.DataFrame(price_data)

print("Bronze Layer DataFrames created:")
print(f"\nNews DataFrame shape: {df_news_bronze.shape}")
print(f"Price DataFrame shape: {df_price_bronze.shape}")
print(f"\nNews columns: {list(df_news_bronze.columns)}")
print(f"\nPrice columns: {list(df_price_bronze.columns)}")


## 2. Data Curation (Silver Layer)

Clean and prepare the data:
- Remove duplicates
- Normalize timestamps
- Filter for English language and AAPL ticker
- Extract relevant fields


In [ ]:
# Process news data (Silver Layer)
df_news_silver = df_news_bronze.copy()

# Convert timestamps
df_news_silver['created'] = pd.to_datetime(df_news_silver['created'], errors='coerce')
df_news_silver['updated'] = pd.to_datetime(df_news_silver['updated'], errors='coerce')

# Use created date as primary timestamp
df_news_silver['date'] = df_news_silver['created'].dt.date
df_news_silver['datetime'] = df_news_silver['created']

# Remove rows with missing timestamps
df_news_silver = df_news_silver.dropna(subset=['created', 'title', 'body'])

# Combine title and body for sentiment analysis
df_news_silver['text'] = df_news_silver['title'].fillna('') + ' ' + df_news_silver['body'].fillna('')
df_news_silver['text'] = df_news_silver['text'].str.strip()

# Remove empty text
df_news_silver = df_news_silver[df_news_silver['text'].str.len() > 0]

# Deduplication: Remove duplicates based on title hash
df_news_silver['title_hash'] = df_news_silver['title'].apply(lambda x: hash(str(x)))
df_news_silver = df_news_silver.drop_duplicates(subset=['title_hash'], keep='first')

# Limit analysis to at most 100 articles (for faster FinBERT inference)
df_news_silver = df_news_silver.sort_values('date').head(100).reset_index(drop=True)

print(f"Silver Layer News: {len(df_news_silver)} articles after cleaning (capped at 100)")
print(f"Date range: {df_news_silver['date'].min()} to {df_news_silver['date'].max()}")


In [ ]:
# Process price data (Silver Layer)
from datetime import datetime

def safe_parse_price_timestamp(ts):
    """Safely convert price timestamp to Python datetime."""
    if pd.isna(ts):
        return None
    ts_str = str(ts).strip()
    
    # Try ISO format first (e.g., '2016-01-04T05:00:00Z')
    iso_formats = [
        "%Y-%m-%dT%H:%M:%S.%fZ",
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d"
    ]
    for fmt in iso_formats:
        try:
            return datetime.strptime(ts_str, fmt)
        except Exception:
            continue
    
    # Try numeric (Unix timestamp - milliseconds or seconds)
    try:
        ts_num = float(ts_str)
        if ts_num > 1e12:  # milliseconds
            return datetime.fromtimestamp(ts_num / 1000)
        else:  # seconds
            return datetime.fromtimestamp(ts_num)
    except Exception:
        return None

df_price_silver = df_price_bronze.copy()

# Map price columns (assuming: c=close, h=high, l=low, o=open, t=timestamp, v=volume)
df_price_silver = df_price_silver.rename(columns={
    'c': 'close',
    'h': 'high',
    'l': 'low',
    'o': 'open',
    't': 'timestamp',
    'v': 'volume'
})

# Convert timestamp using helper function
if 'timestamp' in df_price_silver.columns:
    df_price_silver['datetime'] = df_price_silver['timestamp'].apply(safe_parse_price_timestamp)
    df_price_silver['date'] = pd.to_datetime(df_price_silver['datetime']).dt.date
else:
    # If no timestamp, create a synthetic date index
    df_price_silver['date'] = pd.date_range(start='2020-01-01', periods=len(df_price_silver), freq='D').date

# Remove rows with missing data
df_price_silver = df_price_silver.dropna(subset=['close', 'date'])

# Keep only dates that appear in the (capped) news set
news_dates = set(df_news_silver['date'])
df_price_silver = df_price_silver[df_price_silver['date'].isin(news_dates)]

# Sort by date
df_price_silver = df_price_silver.sort_values('date')

print(f"Silver Layer Price (filtered to news dates): {len(df_price_silver)} records")
print(f"Date range: {df_price_silver['date'].min()} to {df_price_silver['date'].max()}")
print(f"\nPrice data sample:")
print(df_price_silver[['date', 'open', 'high', 'low', 'close', 'volume']].head())


## 3. FinBERT Sentiment Inference

Load FinBERT model and generate sentiment scores for each news article.


In [ ]:
# Load FinBERT model and tokenizer
print("Loading FinBERT model...")
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"FinBERT model loaded on {device}")


In [ ]:
# Function to get sentiment scores
def get_sentiment_scores(text, tokenizer, model, device, max_length=512):
    """
    Get sentiment scores using FinBERT.
    Returns: p_pos, p_neu, p_neg, sentiment_score, confidence
    """
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0
    
    # Truncate text if too long
    text = str(text)[:5000]  # Limit text length
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
    
    # FinBERT labels: positive, neutral, negative
    p_pos = probs[0][0].item()
    p_neu = probs[0][1].item()
    p_neg = probs[0][2].item()
    
    # Calculate sentiment score and confidence
    sentiment_score = p_pos - p_neg
    confidence = max(p_pos, p_neu, p_neg)
    
    return p_pos, p_neu, p_neg, sentiment_score, confidence

print("Sentiment function defined")


In [ ]:
# Apply sentiment analysis to news articles
# Note: This may take a while for large datasets
# For demonstration, we'll process a sample first, then full dataset

print("Processing sentiment for news articles...")
print(f"Total articles to process: {len(df_news_silver)}")

# Process in batches to avoid memory issues
batch_size = 32
sentiment_results = []

for i in range(0, len(df_news_silver), batch_size):
    batch = df_news_silver.iloc[i:i+batch_size]
    batch_results = []
    
    for idx, row in batch.iterrows():
        try:
            p_pos, p_neu, p_neg, sent_score, conf = get_sentiment_scores(
                row['text'], tokenizer, model, device
            )
            batch_results.append({
                'id': row.get('id', idx),
                'p_pos': p_pos,
                'p_neu': p_neu,
                'p_neg': p_neg,
                'sentiment_score': sent_score,
                'confidence': conf
            })
        except Exception as e:
            print(f"Error processing article {idx}: {e}")
            batch_results.append({
                'id': row.get('id', idx),
                'p_pos': 0.0,
                'p_neu': 1.0,
                'p_neg': 0.0,
                'sentiment_score': 0.0,
                'confidence': 0.0
            })
    
    sentiment_results.extend(batch_results)
    
    if (i + batch_size) % 1000 == 0:
        print(f"Processed {i + batch_size} articles...")

print(f"Sentiment analysis complete for {len(sentiment_results)} articles")


In [ ]:
# Merge sentiment results with news data
df_sentiment = pd.DataFrame(sentiment_results)
df_news_silver = df_news_silver.reset_index(drop=True)
df_news_silver = pd.concat([df_news_silver, df_sentiment[['p_pos', 'p_neu', 'p_neg', 'sentiment_score', 'confidence']]], axis=1)

print("Sentiment scores merged with news data")
print(f"\nSentiment statistics:")
print(df_news_silver[['p_pos', 'p_neu', 'p_neg', 'sentiment_score', 'confidence']].describe())


## 4. Daily Aggregation (Gold Layer)

Aggregate sentiment scores into daily metrics and merge with price data.


In [ ]:
# Aggregate sentiment by date (Gold Layer)
# Basic aggregates
df_daily_sentiment = df_news_silver.groupby('date').agg({
    'sentiment_score': ['mean', 'std', 'count'],
    'p_pos': 'mean',
    'p_neu': 'mean',
    'p_neg': 'mean',
    'confidence': 'mean'
}).reset_index()

# Flatten column names
df_daily_sentiment.columns = [
    'date',
    'sent_mean', 'sent_std', 'doc_count',
    'p_pos_mean', 'neu_ratio', 'neg_ratio', 'conf_mean'
]

# Positive article flag for each news item
df_news_silver['is_pos'] = df_news_silver['p_pos'] > df_news_silver['p_neg']

# Compute positive ratio per date (proportion of positive articles)
pos_ratio = (
    df_news_silver
    .groupby('date')['is_pos']
    .mean()
    .reset_index(name='pos_ratio')
)

# Merge positive ratio into daily sentiment and drop temporary column
df_daily_sentiment = (
    df_daily_sentiment
    .merge(pos_ratio, on='date', how='left')
    .drop(columns=['p_pos_mean'])
)

print(f"Daily sentiment metrics: {len(df_daily_sentiment)} days")
print(df_daily_sentiment.head(10))


In [ ]:
# Calculate price metrics
df_price_silver['ret_1d'] = np.log(df_price_silver['close'] / df_price_silver['close'].shift(1))
df_price_silver['vol_1d'] = df_price_silver['ret_1d'].abs()

# Aggregate price data by date (if multiple records per day)
df_daily_price = df_price_silver.groupby('date').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'ret_1d': 'last',  # Use last return of the day
    'vol_1d': 'last'   # Use last volatility of the day
}).reset_index()

print(f"Daily price metrics: {len(df_daily_price)} days")
print(df_daily_price.head(10))


In [ ]:
# Merge sentiment and price data (Gold Layer)
df_gold = pd.merge(
    df_daily_sentiment,
    df_daily_price,
    on='date',
    how='inner'
)

# Ensure date is datetime for proper sorting
df_gold['date'] = pd.to_datetime(df_gold['date'])
df_gold = df_gold.sort_values('date').reset_index(drop=True)

print(f"Gold Layer: {len(df_gold)} days with both sentiment and price data")
print(f"Date range: {df_gold['date'].min()} to {df_gold['date'].max()}")
print(f"\nGold Layer sample:")
print(df_gold[['date', 'sent_mean', 'doc_count', 'close', 'ret_1d', 'vol_1d']].head(10))


## 5. Evaluation: Correlation and Co-movement Analysis

Analyze the relationship between sentiment metrics and market indicators.


In [ ]:
# Calculate correlations
sentiment_vars = ['sent_mean', 'pos_ratio', 'doc_count']
market_vars = ['ret_1d', 'vol_1d']

correlation_results = []

for sent_var in sentiment_vars:
    for market_var in market_vars:
        # Remove NaN values for correlation
        valid_data = df_gold[[sent_var, market_var]].dropna()
        
        if len(valid_data) > 10:  # Need sufficient data points
            pearson_corr, pearson_p = stats.pearsonr(valid_data[sent_var], valid_data[market_var])
            spearman_corr, spearman_p = stats.spearmanr(valid_data[sent_var], valid_data[market_var])
            
            correlation_results.append({
                'sentiment_metric': sent_var,
                'market_metric': market_var,
                'pearson_corr': pearson_corr,
                'pearson_p': pearson_p,
                'spearman_corr': spearman_corr,
                'spearman_p': spearman_p,
                'n_observations': len(valid_data)
            })

df_correlations = pd.DataFrame(correlation_results)

print("Correlation Analysis Results:")
print("=" * 80)
for _, row in df_correlations.iterrows():
    print(f"\n{row['sentiment_metric']} vs {row['market_metric']}:")
    print(f"  Pearson correlation: {row['pearson_corr']:.4f} (p-value: {row['pearson_p']:.4f})")
    print(f"  Spearman correlation: {row['spearman_corr']:.4f} (p-value: {row['spearman_p']:.4f})")
    print(f"  Observations: {row['n_observations']}")


In [ ]:
# Create correlation heatmap
corr_matrix = df_gold[sentiment_vars + market_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Sentiment Metrics vs Market Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Time series visualization: Sentiment vs Returns
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Sentiment Score over time
ax1 = axes[0]
ax1.plot(df_gold['date'], df_gold['sent_mean'], color='blue', alpha=0.7, linewidth=1.5, label='Mean Sentiment Score')
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax1.set_ylabel('Sentiment Score', fontsize=11)
ax1.set_title('Daily Mean Sentiment Score Over Time', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Returns over time
ax2 = axes[1]
ax2.plot(df_gold['date'], df_gold['ret_1d'], color='green', alpha=0.7, linewidth=1.5, label='Daily Returns')
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_ylabel('Log Returns', fontsize=11)
ax2.set_title('AAPL Daily Returns Over Time', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Volatility over time
ax3 = axes[2]
ax3.plot(df_gold['date'], df_gold['vol_1d'], color='red', alpha=0.7, linewidth=1.5, label='Daily Volatility')
ax3.set_ylabel('Absolute Returns (Volatility)', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.set_title('AAPL Daily Volatility Over Time', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots: Sentiment vs Market Metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sentiment vs Returns
ax1 = axes[0, 0]
ax1.scatter(df_gold['sent_mean'], df_gold['ret_1d'], alpha=0.5, s=30)
ax1.set_xlabel('Mean Sentiment Score', fontsize=11)
ax1.set_ylabel('Daily Returns', fontsize=11)
ax1.set_title('Sentiment Score vs Daily Returns', fontsize=12, fontweight='bold')
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax1.axvline(x=0, color='black', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.3)
# Add trend line
z = np.polyfit(df_gold['sent_mean'].dropna(), df_gold['ret_1d'].dropna(), 1)
p = np.poly1d(z)
ax1.plot(df_gold['sent_mean'].dropna(), p(df_gold['sent_mean'].dropna()), "r--", alpha=0.8, linewidth=2)

# Sentiment vs Volatility
ax2 = axes[0, 1]
ax2.scatter(df_gold['sent_mean'], df_gold['vol_1d'], alpha=0.5, s=30, color='orange')
ax2.set_xlabel('Mean Sentiment Score', fontsize=11)
ax2.set_ylabel('Daily Volatility', fontsize=11)
ax2.set_title('Sentiment Score vs Daily Volatility', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', alpha=0.3)
ax2.grid(True, alpha=0.3)
# Add trend line
z = np.polyfit(df_gold['sent_mean'].dropna(), df_gold['vol_1d'].dropna(), 1)
p = np.poly1d(z)
ax2.plot(df_gold['sent_mean'].dropna(), p(df_gold['sent_mean'].dropna()), "r--", alpha=0.8, linewidth=2)

# Positive Ratio vs Returns
ax3 = axes[1, 0]
ax3.scatter(df_gold['pos_ratio'], df_gold['ret_1d'], alpha=0.5, s=30, color='green')
ax3.set_xlabel('Positive Ratio', fontsize=11)
ax3.set_ylabel('Daily Returns', fontsize=11)
ax3.set_title('Positive Ratio vs Daily Returns', fontsize=12, fontweight='bold')
ax3.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax3.grid(True, alpha=0.3)
# Add trend line
z = np.polyfit(df_gold['pos_ratio'].dropna(), df_gold['ret_1d'].dropna(), 1)
p = np.poly1d(z)
ax3.plot(df_gold['pos_ratio'].dropna(), p(df_gold['pos_ratio'].dropna()), "r--", alpha=0.8, linewidth=2)

# Document Count vs Volatility
ax4 = axes[1, 1]
ax4.scatter(df_gold['doc_count'], df_gold['vol_1d'], alpha=0.5, s=30, color='purple')
ax4.set_xlabel('Document Count (News Volume)', fontsize=11)
ax4.set_ylabel('Daily Volatility', fontsize=11)
ax4.set_title('News Volume vs Daily Volatility', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)
# Add trend line
z = np.polyfit(df_gold['doc_count'].dropna(), df_gold['vol_1d'].dropna(), 1)
p = np.poly1d(z)
ax4.plot(df_gold['doc_count'].dropna(), p(df_gold['doc_count'].dropna()), "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.show()


In [ ]:
# Overlay plot: Sentiment and Returns on same axis
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot sentiment (left y-axis)
color = 'tab:blue'
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Mean Sentiment Score', color=color, fontsize=12)
ax1.plot(df_gold['date'], df_gold['sent_mean'], color=color, alpha=0.7, linewidth=2, label='Sentiment Score')
ax1.axhline(y=0, color=color, linestyle='--', alpha=0.3)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Plot returns (right y-axis)
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Daily Returns', color=color, fontsize=12)
ax2.plot(df_gold['date'], df_gold['ret_1d'], color=color, alpha=0.7, linewidth=2, label='Daily Returns')
ax2.axhline(y=0, color=color, linestyle='--', alpha=0.3)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Sentiment Score and Daily Returns Co-movement Over Time', fontsize=14, fontweight='bold', pad=20)
fig.tight_layout()
plt.show()


In [ ]:
# Summary statistics
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\n1. Sentiment Metrics:")
print(df_gold[['sent_mean', 'pos_ratio', 'doc_count', 'conf_mean']].describe())

print("\n2. Market Metrics:")
print(df_gold[['close', 'ret_1d', 'vol_1d', 'volume']].describe())

print("\n3. Key Correlations:")
print(df_correlations[['sentiment_metric', 'market_metric', 'pearson_corr', 'spearman_corr']].to_string(index=False))

print("\n4. Data Coverage:")
print(f"Total days with data: {len(df_gold)}")
print(f"Date range: {df_gold['date'].min()} to {df_gold['date'].max()}")
print(f"Average articles per day: {df_gold['doc_count'].mean():.2f}")
print(f"Total articles analyzed: {df_gold['doc_count'].sum():.0f}")


## 6. Conclusions

This analysis demonstrates:

1. **Data Pipeline**: Successfully implemented Bronze → Silver → Gold architecture for financial sentiment analysis
2. **FinBERT Integration**: Applied FinBERT model to generate sentiment scores for AAPL news articles
3. **Daily Aggregation**: Created daily sentiment metrics (mean sentiment, positive ratio, document count)
4. **Correlation Analysis**: Quantified relationships between sentiment and market indicators (returns, volatility)
5. **Visualization**: Provided clear visualizations of sentiment-price co-movement

### Key Findings:
- The correlation analysis reveals the strength and direction of relationships between news sentiment and market performance
- Time-series visualizations show how sentiment and returns co-move over time
- Scatter plots help identify patterns and outliers in the sentiment-market relationship

### Next Steps:
- Implement lagged correlation analysis (sentiment predicting future returns)
- Add more sophisticated volatility measures (e.g., GARCH models)
- Extend analysis to include other sentiment metrics (e.g., sentiment volatility)
- Implement rolling window correlations to capture time-varying relationships
